###### Loading Libraries and Datasets

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# Load Data
train = pd.read_csv('data/train_cleaned.csv')
test = pd.read_csv('data/test_cleaned.csv')

target_col = 'total_sales'
id_col = 'id'
train_len = len(train)

df = pd.concat([train.drop(columns=[target_col], errors='ignore'), test], ignore_index=True)




###### Feature Engineering

In [2]:
# Zero Shelf Visibility Correction
visibility_mean = df[df['shelf_visibility'] > 0].groupby('product_code')['shelf_visibility'].mean()
df['shelf_visibility_clean'] = df.apply(
    lambda row: visibility_mean.get(row['product_code'], df['shelf_visibility'].mean()) 
    if row['shelf_visibility'] == 0 else row['shelf_visibility'], 
    axis=1
)

# Prefix & Fat Content Normalization
df['product_type_prefix'] = df['product_code'].str[:2]
df.loc[df['product_type_prefix'] == 'NC', 'fat_content'] = 'Non-Edible'

# Frequency & Density Encodings
df['product_code_freq'] = df['product_code'].map(df['product_code'].value_counts())
df['store_code_freq'] = df['store_code'].map(df['store_code'].value_counts())

# Group-Level Price & Visibility Ratios
df['price_vs_cat_mean'] = df['product_price'] / df.groupby('product_category')['product_price'].transform('mean')
df['price_vs_format_mean'] = df['product_price'] / df.groupby('store_format')['product_price'].transform('mean')
df['price_vs_tier_mean'] = df['product_price'] / df.groupby('store_location_tier')['product_price'].transform('mean')
df['price_vs_prefix_mean'] = df['product_price'] / df.groupby('product_type_prefix')['product_price'].transform('mean')

df['vis_vs_cat_mean'] = df['shelf_visibility_clean'] / df.groupby('product_category')['shelf_visibility_clean'].transform('mean')
df['vis_vs_store_mean'] = df['shelf_visibility_clean'] / df.groupby('store_code')['shelf_visibility_clean'].transform('mean')

# Non-Linear Polynomial & Elasticity Features
df['log_product_price'] = np.log1p(df['product_price'])
df['price_squared'] = df['product_price'] ** 2
df['visibility_squared'] = df['shelf_visibility_clean'] ** 2
df['price_x_visibility'] = df['product_price'] * df['shelf_visibility_clean']

# Store Revenue Potential & Sales Density
df['store_price_sum'] = df.groupby('store_code')['product_price'].transform('sum')
df['store_price_mean'] = df.groupby('store_code')['product_price'].transform('mean')
df['store_price_std'] = df.groupby('store_code')['product_price'].transform('std')
df['price_to_store_sum_ratio'] = df['product_price'] / (df['store_price_sum'] + 1e-5)

df['product_store_count'] = df.groupby('product_code')['store_code'].transform('nunique')
df['store_product_count'] = df.groupby('store_code')['product_code'].transform('nunique')

cat_cols = [
    'product_code', 'fat_content', 'product_category', 'product_type_prefix',
    'store_code', 'store_size', 'store_location_tier', 'store_format'
]

for col in cat_cols:
    df[col] = df[col].astype(str)


###### Data Splitting and Model Training

In [3]:
X_train = df.iloc[:train_len].copy()
X_test = df.iloc[train_len:].copy()

y_train_sqrt = np.sqrt(train[target_col])
feature_cols = [c for c in X_train.columns if c != id_col]

* K-Fold Cross-Validation (CV)

In [4]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

oof_m1 = np.zeros(len(X_train))
oof_m2 = np.zeros(len(X_train))
oof_m3 = np.zeros(len(X_train))

test_m1 = np.zeros(len(X_test))
test_m2 = np.zeros(len(X_test))
test_m3 = np.zeros(len(X_test))

print("Starting Triple-Architecture CatBoost Engine...\n")

for fold, (trn_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, y_tr = X_train.iloc[trn_idx][feature_cols], y_train_sqrt.iloc[trn_idx]
    X_va, y_va = X_train.iloc[val_idx][feature_cols], y_train_sqrt.iloc[val_idx]

    # Deep Tree Architecture 
    cat1 = CatBoostRegressor(
        iterations=2800, learning_rate=0.015, depth=8, l2_leaf_reg=5,
        subsample=0.85, cat_features=cat_cols, thread_count=-1, random_seed=42 + fold, verbose=0
    )
    cat1.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=150)
    oof_m1[val_idx] = cat1.predict(X_va)
    test_m1 += cat1.predict(X_test[feature_cols]) / kf.n_splits

    # Medium Tree Architecture 
    cat2 = CatBoostRegressor(
        iterations=3000, learning_rate=0.018, depth=6, l2_leaf_reg=4,
        subsample=0.80, cat_features=cat_cols, thread_count=-1, random_seed=100 + fold, verbose=0
    )
    cat2.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=150)
    oof_m2[val_idx] = cat2.predict(X_va)
    test_m2 += cat2.predict(X_test[feature_cols]) / kf.n_splits

    # Shallow Regularized Architecture 
    cat3 = CatBoostRegressor(
        iterations=3200, learning_rate=0.02, depth=4, l2_leaf_reg=6,
        subsample=0.75, cat_features=cat_cols, thread_count=-1, random_seed=200 + fold, verbose=0
    )
    cat3.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=150)
    oof_m3[val_idx] = cat3.predict(X_va)
    test_m3 += cat3.predict(X_test[feature_cols]) / kf.n_splits

    print(f"Fold {fold + 1}/10 complete.")


# Out-of-Fold Evaluation & Quantile Matching 
pred_m1 = np.square(np.maximum(0, oof_m1))
pred_m2 = np.square(np.maximum(0, oof_m2))
pred_m3 = np.square(np.maximum(0, oof_m3))

# Weighted Ensemble 
ensemble_oof_raw = (pred_m1 * 0.40) + (pred_m2 * 0.45) + (pred_m3 * 0.15)
test_preds_raw = (np.square(np.maximum(0, test_m1)) * 0.40) + \
                 (np.square(np.maximum(0, test_m2)) * 0.45) + \
                 (np.square(np.maximum(0, test_m3)) * 0.15)

# Mean and Variance Calibration 
actual_sales = train[target_col].values
scale_factor = np.mean(actual_sales) / np.mean(ensemble_oof_raw)

calibrated_oof = ensemble_oof_raw * scale_factor
final_test_preds = np.clip(test_preds_raw * scale_factor, 0, None)

score_m1 = np.sqrt(mean_squared_error(actual_sales, pred_m1))
score_m2 = np.sqrt(mean_squared_error(actual_sales, pred_m2))
score_m3 = np.sqrt(mean_squared_error(actual_sales, pred_m3))
score_ensemble = np.sqrt(mean_squared_error(actual_sales, calibrated_oof))

print("\nLOCAL VALIDATION SCORES")
print(f"CatBoost Depth 8 RMSE : {score_m1:.4f}")
print(f"CatBoost Depth 6 RMSE : {score_m2:.4f}")
print(f"CatBoost Depth 4 RMSE : {score_m3:.4f}")
print(f"Calibrated Ensemble RMSE : {score_ensemble:.4f}")



Starting Triple-Architecture CatBoost Engine...

Fold 1/10 complete.
Fold 2/10 complete.
Fold 3/10 complete.
Fold 4/10 complete.
Fold 5/10 complete.
Fold 6/10 complete.
Fold 7/10 complete.
Fold 8/10 complete.
Fold 9/10 complete.
Fold 10/10 complete.

LOCAL VALIDATION SCORES
CatBoost Depth 8 RMSE : 1081.7221
CatBoost Depth 6 RMSE : 1080.3228
CatBoost Depth 4 RMSE : 1079.0834
Calibrated Ensemble RMSE : 1072.5263


* Optimizing Submission File

In [5]:
squeezed_oof_raw = (pred_m1 * 0.10) + (pred_m2 * 0.40) + (pred_m3 * 0.50)

# Test predictions re-weighted
squeezed_test_raw = (np.square(np.maximum(0, test_m1)) * 0.10) + \
                    (np.square(np.maximum(0, test_m2)) * 0.40) + \
                    (np.square(np.maximum(0, test_m3)) * 0.50)

# Calibration
actual_sales = train[target_col].values
scale_factor = np.mean(actual_sales) / np.mean(squeezed_oof_raw)

calibrated_oof_squeezed = squeezed_oof_raw * scale_factor
final_test_preds_squeezed = np.clip(squeezed_test_raw * scale_factor, 0, None)

squeezed_rmse = np.sqrt(mean_squared_error(actual_sales, calibrated_oof_squeezed))

print("\nRE-WEIGHTED VALIDATION SCORE")
print(f"Shallow-Shifted Calibrated Ensemble RMSE : {squeezed_rmse:.4f}")


RE-WEIGHTED VALIDATION SCORE
Shallow-Shifted Calibrated Ensemble RMSE : 1071.8595


 * Saving Final Submission

In [6]:
submission = pd.DataFrame({
    'id': test['id'],
    'total_sales': final_test_preds_squeezed
})

submission.to_csv('submission.csv', index=False)
print("\nUpdated submission saved to 'submission.csv'!")


Updated submission saved to 'submission.csv'!
